# Bổ sung cache chuyển tự + dịch máy cho split chính thức

`data/xlit_kn.json` và `data/mt_en.json` được sinh từ `data/raw` **trước khi** có split chính
thức, nên chỉ phủ 368/806 câu test Task A (365/396 Task B). Notebook này tải split chính thức từ
Google Drive (CSV không nằm trong git), rồi bổ sung đúng **438 câu** còn thiếu, cùng cấu hình như
lần đầu: IndicXlit beam 4, rồi NLLB-200 600M dịch từ chữ Kannada, 4 beam. Câu đã có trong cache
được giữ nguyên.

**Settings:** Accelerator = **GPU T4** · Internet = **On**

In [ ]:
import os, subprocess, sys
REPO, BRANCH, WORK = "trong5nhan6/Text", "main", "/kaggle/working"

TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("GH_TOKEN")
except Exception:
    pass
url = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
hide = (lambda s: s.replace(TOKEN, "***")) if TOKEN else (lambda s: s)

os.chdir(WORK)
cmd = (["git", "-C", "repo", "pull", "--ff-only"] if os.path.isdir("repo/.git")
       else ["git", "clone", "--depth", "1", "-b", BRANCH, url, "repo"])
r = subprocess.run(cmd, capture_output=True, text=True)
print(hide((r.stdout + r.stderr).strip()))
os.chdir(f"{WORK}/repo"); sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

## 0) Tải split chính thức từ Google Drive

Giải nén vào `data/HASTIKA_task splits/`, đúng chỗ `src/data/fill_official_caches.py` tìm.

In [ ]:
!pip -q install gdown
import glob, shutil, zipfile
DRIVE_ID = "1vYVv6rOQFbKV5i4myx97FwVSng8YbipK"
!gdown -q "https://drive.google.com/uc?id={DRIVE_ID}" -O /kaggle/working/splits.zip

dst = "data/HASTIKA_task splits"
os.makedirs(dst, exist_ok=True)
with zipfile.ZipFile("/kaggle/working/splits.zip") as z:
    z.extractall("/kaggle/working/splits_raw")
for p in glob.glob("/kaggle/working/splits_raw/**/*.csv", recursive=True):
    shutil.copy(p, dst)
print(sorted(os.listdir(dst)))

## 1) Danh sách câu còn thiếu

Ghi ra `/kaggle/working/todo_official.json`. Phải là **438 câu**.

In [ ]:
!pip -q install ftfy
!python -m src.data.fill_official_caches --step todo

## 2) Chuyển tự trong môi trường Python 3.9

Kaggle đang chạy **Python 3.13**, và cả `fairseq` lẫn `fairseq-fixed` đều không build được ở đó.
Nên bước này chạy trong một venv **Python 3.9** riêng (tạo bằng `uv`), nơi PyPI có sẵn **wheel
dựng sẵn** của `fairseq==0.12.2`, khỏi phải build:

- `pip<24.1`: `omegaconf 2.0.6` (fairseq kéo theo) có metadata cũ mà pip mới từ chối.
- `torch 1.13.1` bản CPU: 438 câu thì CPU là đủ, và tránh vướng CUDA trong venv.
- `numpy<2`: wheel fairseq được build với numpy 1.x.
- `ai4bharat-transliteration` vẫn cài `--no-deps` như notebook cũ, để không kéo `urduhack` /
  TensorFlow.

Mất khoảng 3 đến 5 phút để cài, phần chuyển tự chỉ vài phút.

In [ ]:
!pip -q install uv
!uv python install 3.9 -q
!uv venv --python 3.9 --seed -q /kaggle/working/py39
PY39 = "/kaggle/working/py39/bin/python"
!{PY39} -m pip -q install "pip<24.1"
!{PY39} -m pip -q install torch==1.13.1+cpu torchaudio==0.13.1+cpu --extra-index-url https://download.pytorch.org/whl/cpu
!{PY39} -m pip -q install "numpy<2" fairseq==0.12.2
!{PY39} -m pip -q install --no-deps ai4bharat-transliteration
!{PY39} -m pip -q install pydload indic-nlp-library ujson sacremoses
!{PY39} -c "import fairseq, torch; print('fairseq', fairseq.__version__, '| torch', torch.__version__)"

In [ ]:
!{PY39} -m src.data.fill_official_caches --step xlit

## 3) Dịch bằng NLLB (môi trường chính, GPU T4)

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "khong co")
!python -m src.data.fill_official_caches --step mt

## 4) Kiểm tra bằng mắt, rồi đóng gói

Cell đóng gói phải in **thiếu sau khi chạy: xlit 0, mt 0**; nếu chưa đủ thì nó dừng và không
tạo zip.

In [ ]:
import json
import pandas as pd
from src.data import transliterate, translate

todo = json.load(open("/kaggle/working/todo_official.json", encoding="utf-8"))
x, m = transliterate.load_cache(), translate.load_cache()
pd.set_option("display.max_colwidth", 60)
display(pd.DataFrame([{"goc": t[:45], "kannada": x.get(t, "?")[:35], "english": m.get(t, "?")[:55]}
                      for t in todo[:10]]))

In [ ]:
!python -m src.data.fill_official_caches --step zip

## 5) Tải về

Tải **`fill_official_caches.zip`** từ panel **Output** và giải nén ở **thư mục gốc của repo**:
zip chứa `data/xlit_kn.json` và `data/mt_en.json`, nên hai file đè đúng chỗ.

In [ ]:
import zipfile
z = "/kaggle/working/fill_official_caches.zip"
print(zipfile.ZipFile(z).namelist())
!ls -lh {z}